# 03 — Train DeepScalper (V2: BTC/USD Crypto)
Trains `DeepScalperNet` (BDQ Double-DQN) on BTC/USD using TradeMaster-aligned hyperparameters.

**V2 CHANGES:**
- Crypto universe: `BTC/USD` (was S&P 100)
- Action space: `Discrete(2)` — FLAT/LONG (no short selling)
- Split: 70/10/20 train/val/test (was 80/20)
- Annualisation: `sqrt(525,960)` for 24/7 crypto (was equity session)
- Reward: hindsight bonus already baked into `env._compute_reward()` — no separate call needed
- Gate: model rejected if test Sharpe < 0.5

**Training loop:** 300 episodes per pair, early stop if val Sharpe stalls for 30 eps.

**Input:**  `/content/drive/MyDrive/algo_trader/data/raw/BTC_USD.parquet`  
**Output:** `/content/drive/MyDrive/algo_trader/weights/BTC_USD.pth`

> Enable GPU: **Runtime → Change runtime type → T4 GPU**

In [ ]:
!pip install -q torch torchvision gymnasium numpy pandas pyarrow pytz tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys
RAW_DIR     = '/content/drive/MyDrive/algo_trader/data/raw'
WEIGHTS_DIR = '/content/drive/MyDrive/algo_trader/weights'
os.makedirs(WEIGHTS_DIR, exist_ok=True)
print(f'Raw data dir:  {RAW_DIR}')
print(f'Weights dir:   {WEIGHTS_DIR}')

In [ ]:
REPO_URL = 'https://github.com/rohanpatrick568/deepscalper_copilot.git'
REPO_DIR = '/content/deepscalper_copilot'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

# algo_trader on path → enables `from colab.deepscalper.X import Y`
ALGO_DIR = REPO_DIR + '/algo_trader'
if ALGO_DIR not in sys.path:
    sys.path.insert(0, ALGO_DIR)
print('Repo on path ✓')

In [ ]:
import torch
print(f'PyTorch {torch.__version__}')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

# -- Architecture -- must match config.py --------------------------------------
MACRO_DIM     = 11   # Table 2 macro features (unchanged)
LOB_DIM       = 4    # V2: dual-mode micro features
PRIV_DIM      = 2    # private state: (position_flag, unrealized_pnl_pct)
N_DIR         = 2    # binary FLAT/LONG
N_SIZE        = 1    # single size branch (Kelly external)
GRU_HIDDEN    = 128
MACRO_EMBED   = 64
FC_HIDDEN     = 128
LOOKBACK_BARS = 10

# -- Training hyperparameters --------------------------------------------------
# batch_size='auto' uses a memory model to solve for batch size that targets
# a chosen fraction of available GPU memory.
TRAINING_CONFIG = dict(
    n_episodes           = 300,
    lr                   = 1e-4,
    tau                  = 0.005,
    buffer_capacity      = 100_000,
    batch_size           = 'auto',  # auto-computed from available GPU memory
    target_gpu_mem_frac  = 0.92,    # aim to use ~92% of total VRAM
    auto_probe_batch     = 256,     # probe batch for memory slope estimate
    batch_size_min       = 512,
    batch_size_max       = 16384,
    updates_per_step     = 4,
    epsilon_decay        = 20_000,
    early_stop_patience  = 30,
    eval_every           = 5,
    eval_episodes        = 10,
    gradient_clip        = 0.5,
    dropout              = 0.1,
)

CRYPTO_PAIRS = ['BTC/USD']
print(f'Crypto pairs: {CRYPTO_PAIRS}')
print(f'Training config: {TRAINING_CONFIG}')

In [ ]:
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

import torch
from colab.deepscalper.agent import DeepScalperAgent
from colab.deepscalper.architecture import DeepScalperNet
from colab.deepscalper.environment import ScalperEnv
from colab.deepscalper.utils import (
    compute_macro_features,
    compute_micro_features,
    compute_day_starts,
    compute_sharpe,
)

_CRYPTO_ANNUALISE = np.sqrt(525_960)


def compute_episode_sharpe(episode_log_returns: list) -> float:
    rets = np.asarray(episode_log_returns, dtype=np.float64)
    if len(rets) < 5:
        return 0.0
    std = rets.std()
    if std == 0:
        return 0.0
    return float((rets.mean() / std) * _CRYPTO_ANNUALISE)


def evaluate_agent(agent: DeepScalperAgent, env: ScalperEnv, n_episodes: int) -> float:
    saved_epsilon = agent.epsilon
    agent.epsilon = 0.0
    all_log_returns = []
    for _ in range(n_episodes):
        obs, _ = env.reset()
        done = False
        while not done:
            dir_act, _size_act = agent.select_action(obs)
            obs, r, terminated, truncated, info = env.step(dir_act)
            all_log_returns.append(info.get('log_return', r))
            done = terminated or truncated
    agent.epsilon = saved_epsilon
    return compute_episode_sharpe(all_log_returns)


def _gpu_mem_gb() -> float:
    if DEVICE != 'cuda':
        return 0.0
    return float(torch.cuda.memory_allocated() / (1024 ** 3))


def _round_down(x: int, multiple: int) -> int:
    return max(multiple, (x // multiple) * multiple)


def _probe_peak_bytes_for_batch(
    batch_size: int,
    macro_dim: int,
    lob_dim: int,
    priv_dim: int,
    lookback_bars: int,
    n_dir: int,
    n_size: int,
    gru_hidden: int,
    macro_embed: int,
    fc_hidden: int,
) -> int:
    """Measure peak allocated CUDA bytes for one fwd+bwd step at batch_size."""
    if DEVICE != 'cuda':
        return 0

    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    net = DeepScalperNet(
        macro_dim=macro_dim,
        lob_dim=lob_dim,
        priv_dim=priv_dim,
        gru_hidden=gru_hidden,
        macro_embed=macro_embed,
        fc_hidden=fc_hidden,
        n_dir=n_dir,
        n_size=n_size,
    ).to(DEVICE)
    opt = torch.optim.Adam(net.parameters(), lr=1e-4)

    lob = torch.randn(batch_size, lookback_bars, lob_dim, device=DEVICE)
    priv = torch.randn(batch_size, lookback_bars, priv_dim, device=DEVICE)
    macro = torch.randn(batch_size, macro_dim, device=DEVICE)

    q_dir, q_size, vol = net(lob, priv, macro)
    loss = (q_dir.pow(2).mean() + q_size.pow(2).mean() + vol.pow(2).mean())
    opt.zero_grad(set_to_none=True)
    loss.backward()
    opt.step()
    torch.cuda.synchronize()
    peak = int(torch.cuda.max_memory_allocated())

    del lob, priv, macro, q_dir, q_size, vol, loss, opt, net
    torch.cuda.empty_cache()
    return peak


def auto_batch_size(
    macro_dim: int,
    lob_dim: int,
    priv_dim: int,
    lookback_bars: int,
    n_dir: int,
    n_size: int,
    gru_hidden: int,
    macro_embed: int,
    fc_hidden: int,
    target_gpu_mem_frac: float,
    probe_batch: int,
    batch_min: int,
    batch_max: int,
    buffer_capacity: int,
    train_rows: int,
) -> int:
    """Solve for batch size to target a fraction of VRAM using measured slope."""
    if DEVICE != 'cuda':
        return max(batch_min, 256)

    total_bytes = int(torch.cuda.get_device_properties(0).total_memory)
    target_bytes = int(total_bytes * target_gpu_mem_frac)

    baseline = int(torch.cuda.memory_allocated())
    peak_probe = _probe_peak_bytes_for_batch(
        probe_batch, macro_dim, lob_dim, priv_dim, lookback_bars,
        n_dir, n_size, gru_hidden, macro_embed, fc_hidden,
    )

    # Approximate bytes/sample for one train step of the model.
    # Agent learn() does more work than this probe (online+target + buffers),
    # so apply a conservative multiplier.
    usable_probe = max(1, peak_probe - baseline)
    bytes_per_sample = (usable_probe / float(probe_batch)) * 3.2

    available = max(1, target_bytes - baseline)
    estimated = int(available / bytes_per_sample)

    # Cap by practical learning dynamics and buffer constraints.
    dynamic_max = min(batch_max, buffer_capacity // 2, max(batch_min, train_rows // 4))
    estimated = max(batch_min, min(dynamic_max, estimated))

    # Favor tensor-core friendly multiples.
    estimated = _round_down(estimated, 32)
    return max(batch_min, estimated)


# -- Main training loop --------------------------------------------------------
training_log = []
for pair in tqdm(CRYPTO_PAIRS, desc='Pairs'):
    safe_name    = pair.replace('/', '_')
    weights_path = f'{WEIGHTS_DIR}/{safe_name}.pth'

    if os.path.exists(weights_path):
        print(f'{pair}: weights already exist - skipping.')
        continue

    raw_path = f'{RAW_DIR}/{safe_name}.parquet'
    if not os.path.exists(raw_path):
        print(f'WARNING: {raw_path} missing - run 01_fetch_training_data first.')
        continue

    bars = pd.read_parquet(raw_path)
    bars.columns = [c.lower() for c in bars.columns]
    bars = bars[['open', 'high', 'low', 'close', 'volume']].astype(float)

    macro_feats = compute_macro_features(bars)
    lob_feats   = compute_micro_features(bars)
    close_arr   = bars['close'].values.astype(np.float64)
    day_starts  = compute_day_starts(bars.index)

    if len(day_starts) < 14:
        print(f'WARNING: {pair} has only {len(day_starts)} days - need >=14, skipping.')
        continue

    n = len(bars)
    train_end = int(n * 0.70)
    val_end   = int(n * 0.80)

    train_df = bars.iloc[:train_end]
    val_df   = bars.iloc[train_end:val_end]
    test_df  = bars.iloc[val_end:]

    assert train_df.index[-1] < val_df.index[0], 'Train/val overlap detected!'
    assert val_df.index[-1] < test_df.index[0], 'Val/test overlap detected!'

    train_day_starts = [d for d in day_starts if d < train_end]
    val_day_starts   = [d - train_end for d in day_starts if train_end <= d < val_end]
    test_day_starts  = [d - val_end for d in day_starts if d >= val_end]

    if not train_day_starts or not val_day_starts or not test_day_starts:
        print(f'WARNING: {pair} insufficient days in one split - skipping.')
        continue

    train_env = ScalperEnv(
        lob_features=lob_feats[:train_end],
        macro_features=macro_feats[:train_end],
        close_prices=close_arr[:train_end],
        day_starts=train_day_starts,
        lookback_bars=LOOKBACK_BARS,
    )
    val_env = ScalperEnv(
        lob_features=lob_feats[train_end:val_end],
        macro_features=macro_feats[train_end:val_end],
        close_prices=close_arr[train_end:val_end],
        day_starts=val_day_starts,
        lookback_bars=LOOKBACK_BARS,
    )
    test_env = ScalperEnv(
        lob_features=lob_feats[val_end:],
        macro_features=macro_feats[val_end:],
        close_prices=close_arr[val_end:],
        day_starts=test_day_starts,
        lookback_bars=LOOKBACK_BARS,
    )

    if TRAINING_CONFIG['batch_size'] == 'auto':
        resolved_batch_size = auto_batch_size(
            macro_dim=MACRO_DIM,
            lob_dim=LOB_DIM,
            priv_dim=PRIV_DIM,
            lookback_bars=LOOKBACK_BARS,
            n_dir=N_DIR,
            n_size=N_SIZE,
            gru_hidden=GRU_HIDDEN,
            macro_embed=MACRO_EMBED,
            fc_hidden=FC_HIDDEN,
            target_gpu_mem_frac=TRAINING_CONFIG['target_gpu_mem_frac'],
            probe_batch=TRAINING_CONFIG['auto_probe_batch'],
            batch_min=TRAINING_CONFIG['batch_size_min'],
            batch_max=TRAINING_CONFIG['batch_size_max'],
            buffer_capacity=TRAINING_CONFIG['buffer_capacity'],
            train_rows=len(train_df),
        )
    else:
        resolved_batch_size = int(TRAINING_CONFIG['batch_size'])

    print(f'{pair}: resolved batch_size={resolved_batch_size}')

    agent = DeepScalperAgent(
        macro_dim=MACRO_DIM,
        lob_dim=LOB_DIM,
        priv_dim=PRIV_DIM,
        n_dir=N_DIR,
        n_size=N_SIZE,
        gru_hidden=GRU_HIDDEN,
        macro_embed=MACRO_EMBED,
        fc_hidden=FC_HIDDEN,
        device=DEVICE,
        lr=TRAINING_CONFIG['lr'],
        tau=TRAINING_CONFIG['tau'],
        batch_size=resolved_batch_size,
        buffer_capacity=TRAINING_CONFIG['buffer_capacity'],
        epsilon_decay=TRAINING_CONFIG['epsilon_decay'],
    )

    best_sharpe    = -np.inf
    patience_count = 0
    MAX_EPISODES   = TRAINING_CONFIG['n_episodes']
    PATIENCE       = TRAINING_CONFIG['early_stop_patience']
    EVAL_EVERY     = TRAINING_CONFIG['eval_every']
    EVAL_EPISODES  = TRAINING_CONFIG['eval_episodes']
    UPDATES_PER_STEP = TRAINING_CONFIG['updates_per_step']

    episode_pbar = tqdm(range(1, MAX_EPISODES + 1), desc=f'{pair} episodes', leave=False)
    for episode in episode_pbar:
        obs, _ = train_env.reset()
        done = False
        episode_reward = 0.0
        episode_returns = []
        learn_losses = []
        step_count = 0

        while not done:
            dir_act, _size_act = agent.select_action(obs)
            next_obs, reward, terminated, truncated, info = train_env.step(dir_act)
            done = terminated or truncated

            agent.store(
                obs=obs,
                dir_action=dir_act,
                size_action=0,
                reward=reward,
                next_obs=next_obs,
                done=done,
                vol_target=info['vol_target'],
            )

            for _ in range(UPDATES_PER_STEP):
                loss = agent.learn()
                if loss is not None:
                    learn_losses.append(loss)

            obs = next_obs
            episode_reward += float(reward)
            episode_returns.append(info.get('log_return', 0.0))
            step_count += 1

        mean_loss = float(np.mean(learn_losses)) if learn_losses else float('nan')
        episode_sharpe = compute_episode_sharpe(episode_returns)
        postfix = {
            'eps': f'{agent.epsilon:.3f}',
            'steps': step_count,
            'rew': f'{episode_reward:.3f}',
            'shr': f'{episode_sharpe:.3f}',
            'loss': (f'{mean_loss:.4f}' if not np.isnan(mean_loss) else 'warmup'),
        }
        if DEVICE == 'cuda':
            postfix['gpu_gb'] = f'{_gpu_mem_gb():.2f}'
        episode_pbar.set_postfix(postfix)

        if episode % EVAL_EVERY == 0:
            val_sharpe = evaluate_agent(agent, val_env, EVAL_EPISODES)

            if val_sharpe > best_sharpe:
                best_sharpe = val_sharpe
                patience_count = 0
                agent.save(weights_path)
            else:
                patience_count += 1

            eval_postfix = {
                'eps': f'{agent.epsilon:.3f}',
                'val': f'{val_sharpe:.3f}',
                'best': f'{best_sharpe:.3f}',
                'pat': f'{patience_count}/{PATIENCE}',
            }
            if DEVICE == 'cuda':
                eval_postfix['gpu_gb'] = f'{_gpu_mem_gb():.2f}'
            episode_pbar.set_postfix(eval_postfix)

            if patience_count >= PATIENCE:
                print(f'{pair}: early stop at ep {episode} (best val Sharpe={best_sharpe:.3f})')
                break

    episode_pbar.close()

    if not os.path.exists(weights_path):
        agent.save(weights_path)

    print(f'\n{pair}: Running out-of-sample test evaluation...')
    saved_eps = agent.epsilon
    agent.epsilon = 0.0
    test_log_returns = []
    test_positions = []
    trade_durations = []
    current_trade_len = 0

    for _ in range(len(test_day_starts)):
        obs, _ = test_env.reset()
        done = False
        while not done:
            dir_act, _size_act = agent.select_action(obs)
            obs, r, terminated, truncated, info = test_env.step(dir_act)
            log_ret = info.get('log_return', 0.0)
            pos = info.get('position', 0)
            test_log_returns.append(log_ret)
            test_positions.append(pos)
            if pos == 1:
                current_trade_len += 1
            elif current_trade_len > 0:
                trade_durations.append(current_trade_len)
                current_trade_len = 0
            done = terminated or truncated

    if current_trade_len > 0:
        trade_durations.append(current_trade_len)

    agent.epsilon = saved_eps

    rets = np.array(test_log_returns)
    cum = np.cumsum(rets)
    peak = np.maximum.accumulate(cum)
    max_dd = float(np.abs(cum - peak).max())

    test_sharpe = compute_episode_sharpe(rets.tolist())
    win_rate = float(np.mean(rets > 0)) if len(rets) > 0 else 0.0
    avg_trade_dur = float(np.mean(trade_durations)) if trade_durations else 0.0

    print(f'  Test Sharpe:         {test_sharpe:.4f}  (target >= 0.5)')
    print(f'  Max Drawdown:        {max_dd*100:.2f}%   (limit <= 15%)')
    print(f'  Win Rate:            {win_rate*100:.1f}%     (target >= 45%)')
    print(f'  Avg Trade Duration:  {avg_trade_dur:.1f} bars  (target >= 3)')

    gate_passed = test_sharpe >= 0.5
    if not gate_passed:
        print(f'  [WARN] GATE FAILED: Sharpe {test_sharpe:.4f} < 0.5 - model NOT accepted.')
        print('      Run 06_sharpe_diagnosis.ipynb to diagnose and improve.')
    else:
        print(f'  [OK] All gates passed - weights saved to {weights_path}')

    training_log.append({
        'pair': pair,
        'best_val_sharpe': round(best_sharpe, 4),
        'test_sharpe': round(test_sharpe, 4),
        'max_drawdown_pct': round(max_dd * 100, 2),
        'win_rate_pct': round(win_rate * 100, 1),
        'avg_trade_duration': round(avg_trade_dur, 1),
        'gate_passed': gate_passed,
        'batch_size': resolved_batch_size,
    })
    print(f'{pair}: best val Sharpe = {best_sharpe:.4f}  |  test Sharpe = {test_sharpe:.4f}')

print('\n=== TRAINING COMPLETE ===')
if training_log:
    df_log = pd.DataFrame(training_log).sort_values('test_sharpe', ascending=False)
    print(df_log.to_string(index=False))

log_path = '/content/drive/MyDrive/algo_trader/training_log.csv'
pd.DataFrame(training_log).to_csv(log_path, index=False)
print(f'\nTraining log saved -> {log_path}')